## Importing Libs

In [2]:
import json
import glob
from tqdm import tqdm
import os

import peft
from peft import LoraConfig, get_peft_model, PeftModel

from pathlib import Path

import torch

import datasets

from dataclasses import dataclass, field

from huggingface_hub import snapshot_download

import transformers

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
)

print(transformers.__version__)
print(peft.__version__)

c:\Users\vasco\Desktop\Uni\Mestrado\2º Ano\1º Semestre\CL\KL_Knowledge-Injection-Hallucinations\venv311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


4.57.3
0.8.2


In [3]:
GENERATION_MODEL = "Qwen/Qwen3-4B-Instruct-2507"

FILES = glob.glob("../Knowledge-graph/doc_*/content_*.txt")

NUM_EPOCHS = 3
BATCH_SIZE = 4
LR = 2e-4
MAX_LENGTH = 512
LORA_R = 8
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
SEED = 42
OUTPUT_DIR = "./qwen34_lora_adapter"

In [4]:
dataset = datasets.load_dataset("text", data_files={"train": FILES})

print("Loaded documents:", len(dataset["train"]))

tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '<|pad|>'})

model = AutoModelForCausalLM.from_pretrained(GENERATION_MODEL, trust_remote_code=True)
model.resize_token_embeddings(len(tokenizer))

Loaded documents: 338


Loading checkpoint shards: 100%|██████████| 3/3 [00:06<00:00,  2.28s/it]


Embedding(151669, 2560)

In [5]:
def tokenize_fn(examples):
    return tokenizer(examples["text"], return_attention_mask=False)

def group_texts(examples):
    concatenated = sum(examples['input_ids'], [])
    total_length = len(concatenated)
    if total_length >= MAX_LENGTH:
        total_length = (total_length // MAX_LENGTH) * MAX_LENGTH
    else:
        total_length = (total_length // MAX_LENGTH + 1) * MAX_LENGTH

    result = {
        'input_ids': [concatenated[i:i + MAX_LENGTH] for i in range(0, total_length, MAX_LENGTH)]
    }

    result['labels'] = [list(seq) for seq in result['input_ids']]
    return result

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized = tokenized.map(group_texts, batched=True)

print("Number of training samples:", len(tokenized["train"]))


data_collaborator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)


lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj"],  # module names for Qwen models
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=1,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    logging_steps=20,
    save_total_limit=2,
    save_steps=200,
    remove_unused_columns=False,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    data_collator=data_collaborator,
)

trainer.train()

os.makedirs(OUTPUT_DIR, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapter saved to {OUTPUT_DIR}")

Map: 100%|██████████| 338/338 [00:00<00:00, 27496.70 examples/s]


Number of training samples: 30


c:\Users\vasco\Desktop\Uni\Mestrado\2º Ano\1º Semestre\CL\KL_Knowledge-Injection-Hallucinations\venv311\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
20,2.122500


c:\Users\vasco\Desktop\Uni\Mestrado\2º Ano\1º Semestre\CL\KL_Knowledge-Injection-Hallucinations\venv311\Lib\site-packages\peft\utils\save_and_load.py:160: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
c:\Users\vasco\Desktop\Uni\Mestrado\2º Ano\1º Semestre\CL\KL_Knowledge-Injection-Hallucinations\venv311\Lib\site-packages\peft\utils\save_and_load.py:160: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


LoRA adapter saved to ./qwen34_lora_adapter
